# Fink/LSST — Temporal dynamics of alerts (HEALPix)

This notebook visualises the **temporal dynamics** of Fink/LSST alerts on HEALPix skymaps:

- Last **24h** — **3 days** — **7 days** — **30 days**
- **Custom date range** (start / end date)
- Annotations: Galactic plane, Galactic centre, Magellanic Clouds, Deep Drilling Fields
- **Animation** of sliding time windows

---
**Dependencies**: `healpy`, `astropy`, `numpy`, `matplotlib`, `requests`, `pandas`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.animation as animation
from matplotlib.colors import LogNorm
import healpy as hp
import requests
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.time import Time
from datetime import datetime, timedelta, timezone
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

print(f'healpy  : {hp.__version__}')
print(f'numpy   : {np.__version__}')
print(f'Current UTC time: {datetime.now(timezone.utc).isoformat()}')

## 1. Configuration

In [ ]:
# ---- HEALPix parameters ----
NSIDE = 64
NPIX  = hp.nside2npix(NSIDE)
print(f'NSIDE={NSIDE}  ->  {NPIX} pixels, resolution ~{hp.nside2resol(NSIDE, arcmin=True):.1f} arcmin')

# ---- Fink LSST API ----
FINK_API = 'https://api.lsst.fink-portal.org'

# ---- Tags used for temporal dynamics ----
TAGS = [
    'extragalactic_lt20mag_candidate',
    'extragalactic_new_candidate',
    'hostless_candidate',
    'in_tns',
    'sn_near_galaxy_candidate',
]

# ---- Predefined time windows (days) ----
TIME_WINDOWS = {
    '24h':       1,
    '3 days':    3,
    '1 week':    7,
    '1 month':  30,
}

# ---- Custom date range (edit here) ----
CUSTOM_START = '2025-01-01 00:00:00'
CUSTOM_STOP  = '2026-03-07 23:59:59'

# ---- Data volume ----
N_PER_TAG = 5000   # alerts per tag

## 2. Astronomical annotations

```
| **Name of Field**    | **RA(Degrees)** | **Dec (Degress)**| **Type**          |
| -------------------- | --------------- | ---------------- | ----------------- |
| **Carina**           | 161.5           | -59.7            | Galaxie/Nébuleuse |
| **ELAIS-S1**         | 9.45            | -44.0            | DDF               |
| **COSMOS**           | 150.1           | +2.2             | DDF               |
| **Trifid-Lagoon**    | 270.5           | -23.0            | Nébuleuse         |
| **M49**              | 187.4           | +8.0             | Galaxie           |
| **Rubin_SV_280_-48** | 280.0           | -48.0            | Test SV           |
| **Rubin_SV_320_-15** | 320.0           | -15.0            | Test SV           |
| **Rubin_SV_216_-17** | 216.0           | -17.0            | Test SV           |
| **Rubin_SV_212_-7**  | 212.0           | -7.0             | Test SV           |
| **Rubin_SV_225_-40** | 225.0           | -40.0            | Test SV           |
```

In [ ]:
def galactic_plane_radec(n_points=500, b_deg=0):
    """Return RA/Dec at Galactic latitude b_deg."""
    l_range = np.linspace(0, 360, n_points) * u.deg
    coords  = SkyCoord(l=l_range, b=np.full(n_points, b_deg)*u.deg, frame='galactic')
    icrs    = coords.icrs
    return icrs.ra.deg, icrs.dec.deg

LANDMARKS = {
    'Galactic centre': SkyCoord(l=0*u.deg, b=0*u.deg, frame='galactic').icrs,
    'LMC': SkyCoord(ra=80.89*u.deg,  dec=-69.76*u.deg, frame='icrs'),
    'SMC': SkyCoord(ra=13.16*u.deg,  dec=-72.80*u.deg, frame='icrs'),
}

DDF = {
    'ELAISS1':                 SkyCoord(ra=9.45*u.deg, dec=-44.02*u.deg,   frame='icrs'),
    'COSMOS':                  SkyCoord(ra=150.1191*u.deg, dec=2.2058*u.deg,   frame='icrs'),
    'XMM-LSS':                 SkyCoord(ra=34.3900*u.deg,  dec=-4.9000*u.deg,  frame='icrs'),
    'ECDFS':                   SkyCoord(ra=53.1250*u.deg,  dec=-28.1000*u.deg, frame='icrs'),
    'EDFS-a':                  SkyCoord(ra=58.9000*u.deg,  dec=-49.3150*u.deg, frame='icrs'),
    'EDFS-b':                  SkyCoord(ra=63.6000*u.deg,  dec=-47.6000*u.deg, frame='icrs'),
    'Euclid Deep Field South': SkyCoord(ra=61.2400*u.deg,  dec=-48.4230*u.deg, frame='icrs'),
}

# Pre-compute Galactic plane curves
_gal_plane_ra,   _gal_plane_dec   = galactic_plane_radec()
_gal_plus10_ra,  _gal_plus10_dec  = galactic_plane_radec(b_deg= 10)
_gal_minus10_ra, _gal_minus10_dec = galactic_plane_radec(b_deg=-10)

print('Annotations defined.')

## 3. API access and processing functions

In [ ]:
def radec_to_healpix(ra_deg, dec_deg, nside=NSIDE):
    """Convert RA/Dec (degrees) to HEALPix pixel indices (RING ordering)."""
    theta = np.radians(90.0 - np.asarray(dec_deg))
    phi   = np.radians(np.asarray(ra_deg))
    return hp.ang2pix(nside, theta, phi)

def fetch_tag_alerts(tag, n=N_PER_TAG,
                     startdate=None, stopdate=None,
                     columns='r:ra,r:dec,r:midpointMjdTai'):
    """
    Fetch Fink alerts for a given tag.
    If startdate/stopdate provided, downloads more alerts and filters locally.
    """
    url    = f'{FINK_API}/api/v1/tags'
    params = {
        'tag': tag,
        'n': n,
        'columns': columns,
        'output-format': 'csv',
    }
    if startdate:
        params['startdate'] = startdate
    if stopdate:
        params['stopdate']  = stopdate

    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text))
    df.columns = [c.replace('r:', '') for c in df.columns]
    return df

def fetch_all_tags(startdate=None, stopdate=None, n=N_PER_TAG):
    """Download alerts for all tags and merge into one DataFrame."""
    dfs = []
    for tag in TAGS:
        try:
            df = fetch_tag_alerts(tag, n=n, startdate=startdate, stopdate=stopdate)
            df['tag'] = tag
            dfs.append(df)
        except Exception as e:
            print(f'  Warning {tag}: {e}')
    if dfs:
        merged = pd.concat(dfs, ignore_index=True)
        merged['datetime_utc'] = Time(merged['midpointMjdTai'].values,
                                       format='mjd', scale='tai'
                                       ).utc.to_datetime(timezone=timezone.utc)
        return merged
    return pd.DataFrame(columns=['ra','dec','midpointMjdTai','tag','datetime_utc'])

def make_skymap_from_df(df, nside=NSIDE):
    """Build a HEALPix skymap from a DataFrame with ra/dec columns."""
    skymap = np.zeros(hp.nside2npix(nside))
    if len(df) == 0:
        return skymap
    mask = np.isfinite(df['ra'].values) & np.isfinite(df['dec'].values)
    pix  = radec_to_healpix(df['ra'].values[mask], df['dec'].values[mask], nside)
    np.add.at(skymap, pix, 1)
    return skymap

print('API functions defined.')

## 4. Download global dataset (last 30 days)

In [ ]:
now_utc  = datetime.now(timezone.utc)
start_30 = (now_utc - timedelta(days=30)).strftime('%Y-%m-%d %H:%M:%S')
stop_now = now_utc.strftime('%Y-%m-%d %H:%M:%S')

print(f'Period: {start_30}  ->  {stop_now} UTC')
print('Downloading...')

df_30days = fetch_all_tags(
    startdate=start_30,
    stopdate=stop_now,
    n=N_PER_TAG
)

print(f'\nTotal alerts downloaded (30d): {len(df_30days):,}')
if len(df_30days) > 0:
    print(df_30days.groupby('tag').size().rename('N alerts').to_string())

## 5. Temporal visualisation function

In [ ]:
# --------------------------------------------------------------------------
# EMPTY_PIX: light grey for zero-count pixels.
# Replaces default black background so coordinate grid is visible.
# --------------------------------------------------------------------------
EMPTY_PIX = '#cccccc'

def _set_cmap_bad(name, bad_color=None):
    """Return a copy of colormap `name` with bad/under pixels -> EMPTY_PIX."""
    import copy, matplotlib.cm as mcm
    bc = bad_color if bad_color is not None else EMPTY_PIX
    cm = copy.copy(mcm.get_cmap(name))
    cm.set_bad(bc)
    cm.set_under(bc)
    return cm

def _add_coord_labels():
    """Overlay RA and Dec tick labels on the current healpy Axes."""
    lc = '#111111'
    for dec in (-60, -30, 30, 60):
        theta = np.radians(90 - dec)
        hp.projtext(theta, 0.02,
                    '{:+d} deg'.format(dec),
                    lonlat=False, color=lc, fontsize=7.5,
                    ha='left', va='center', fontweight='bold')
    for ra in range(0, 360, 60):
        hp.projtext(np.pi / 2, np.radians(ra),
                    '{:d} deg'.format(ra),
                    lonlat=False, color=lc, fontsize=7.5,
                    ha='center', va='bottom', fontweight='bold')

def add_annotations_to_mollview(show_plane=True, show_landmarks=True,
                                  show_ddf=True, alpha_plane=0.95):
    """Add astronomical annotations to the active mollview map."""

    def _split_and_plot(ra_arr, dec_arr, **kwargs):
        idx    = np.argsort(ra_arr)
        ra_s, dec_s = ra_arr[idx], dec_arr[idx]
        jumps  = np.where(np.abs(np.diff(ra_s)) > 180)[0] + 1
        for seg_ra, seg_dec in zip(np.split(ra_s, jumps), np.split(dec_s, jumps)):
            hp.projplot(np.radians(90 - seg_dec), np.radians(seg_ra),
                        lonlat=False, **kwargs)

    if show_plane:
        _split_and_plot(_gal_plane_ra, _gal_plane_dec,
                        color='#0080ff', lw=2.2, ls='-', alpha=alpha_plane)
        _split_and_plot(_gal_plus10_ra, _gal_plus10_dec,
                        color='#0080ff', lw=1.0, ls='--', alpha=0.60)
        _split_and_plot(_gal_minus10_ra, _gal_minus10_dec,
                        color='#0080ff', lw=1.0, ls='--', alpha=0.60)

    if show_landmarks:
        style_map = {
            'Galactic centre': dict(marker='*', color='#e6b800', s=200),
            'LMC':             dict(marker='o', color='#00cc44', s=100),
            'SMC':             dict(marker='o', color='#00cc44', s=70),
        }
        for name, coord_obj in LANDMARKS.items():
            theta = np.radians(90 - coord_obj.dec.deg)
            phi   = np.radians(coord_obj.ra.deg)
            style = style_map.get(name, dict(marker='x', color='#333333', s=60))
            hp.projscatter(theta, phi, lonlat=False, zorder=5, **style)
            hp.projtext(theta, phi, '  ' + name, color=style['color'],
                        fontsize=7, fontweight='bold', lonlat=False)

    if show_ddf:
        for name, coord_obj in DDF.items():
            theta = np.radians(90 - coord_obj.dec.deg)
            phi   = np.radians(coord_obj.ra.deg)
            hp.projscatter(theta, phi, marker='+', color='white',
                           s=60, edgecolors='#ff7700', linewidths=0.5,
                           zorder=6, lonlat=False,alpha=0.5)
            hp.projtext(theta, phi, '  ' + name, color='#ff7700',
                        fontsize=6.5, fontstyle='italic', lonlat=False)

    _add_coord_labels()


def plot_temporal_skymap(df_window, title='', cmap='hot',
                          log_scale=True, figsize=(14, 7),
                          start_str='', stop_str=''):
    """
    HEALPix skymap for a given time window.

    Notes
    -----
    DDFs are shown as markers only (< 1 NSIDE=64 pixel in size).
    """
    skymap = make_skymap_from_df(df_window)
    map_to_plot = np.log10(1 + skymap) if log_scale else skymap.copy()
    map_to_plot[map_to_plot == 0] = hp.UNSEEN

    base_cmap = _set_cmap_bad(cmap)
    n_alerts = len(df_window)
    subtitle  = '{} -> {}  |  N={:,} alerts'.format(start_str, stop_str, n_alerts)

    plt.figure(figsize=figsize)
    hp.mollview(
        map_to_plot,
        cmap=base_cmap,
        title='{}\n{}'.format(title, subtitle),
        unit='log10(1+N)' if log_scale else 'Alert count',
        hold=False,
        margins=(0.01, 0.06, 0.01, 0.04),
        bgcolor=EMPTY_PIX,
        badcolor=EMPTY_PIX,
    )
    hp.graticule(dpar=30, dmer=60, alpha=0.75, color='#444444', lw=1.1, local=False)
    hp.graticule(dpar=10, dmer=30, alpha=0.25, color='#888888', lw=0.4,  local=False)
    add_annotations_to_mollview()

    handles = [
        mpatches.Patch(color='#0080ff', alpha=0.9, label='Galactic plane (b=0 deg)'),
        plt.Line2D([0],[0], color='#0080ff', lw=1.0, ls='--', label='|b|=10 deg'),
        plt.Line2D([0],[0], color='#e6b800', marker='*', ms=11,
                   ls='None', label='Galactic centre'),
        plt.Line2D([0],[0], color='#00cc44', marker='o', ms=7,
                   ls='None', label='LMC / SMC'),
        plt.Line2D([0],[0], color='#ff7700', marker='+', ms=7,
                   ls='None', markerfacecolor='white',
                   markeredgecolor='#ff7700', label='DDF (markers only)'),
        mpatches.Patch(color=EMPTY_PIX, ec='#aaaaaa', label='Unobserved pixels'),
    ]
    #plt.gca().legend(handles=handles, loc='lower right', fontsize=7,
    #                 framealpha=0.90, facecolor='white', edgecolor='#aaaaaa')

    plt.gca().legend(
            handles=handles,
            loc='upper left',
            bbox_to_anchor=(1.01, 1.0),   # à droite du plot
            borderaxespad=0,
            fontsize=10,
            framealpha=0.90,
            facecolor='white',
            edgecolor='#aaaaaa'
        )

    
    plt.figtext(0.12, 0.02,
                'NSIDE={}  |  Tags: {}  |  DDFs = markers'.format(NSIDE, len(TAGS)),
                fontsize=7.5, color='#444444')
    plt.tight_layout()
    plt.show()
    return skymap


print('Visualisation functions defined.')

## 6. Skymaps for predefined time windows

In [ ]:
skymaps_temporal = {}

for window_name, n_days in TIME_WINDOWS.items():
    t_start = now_utc - timedelta(days=n_days)
    mask = (
        (df_30days['datetime_utc'] >= t_start) &
        (df_30days['datetime_utc'] <= now_utc)
    ) if len(df_30days) > 0 else pd.Series([], dtype=bool)

    df_win = df_30days[mask] if len(df_30days) > 0 else df_30days

    print(f'\n=== Window: {window_name} ({n_days} day(s)) ===')
    print(f'  Alerts: {len(df_win):,}')

    sm = plot_temporal_skymap(
        df_win,
        title=f'Fink/LSST -- Last {window_name}',
        cmap='hot',
        log_scale=(len(df_win) > 10),
        start_str=t_start.strftime('%Y-%m-%d %H:%M UTC'),
        stop_str=now_utc.strftime('%Y-%m-%d %H:%M UTC'),
    )
    skymaps_temporal[window_name] = sm

## 7. Comparative panel: 4 time windows

In [ ]:
fig, _ = plt.subplots(2, 2, figsize=(22, 12))
plt.subplots_adjust(hspace=0.01, wspace=0.01)

CMAPS_TEMPORAL = ['hot', 'plasma', 'inferno', 'magma']

for i, (window_name, n_days) in enumerate(TIME_WINDOWS.items()):
    plt.subplot(2, 2, i + 1)

    sm = skymaps_temporal.get(window_name, np.zeros(NPIX))
    map_plot = np.log10(1 + sm)
    map_plot[map_plot == 0] = hp.UNSEEN

    t_start = now_utc - timedelta(days=n_days)
    mask = (
        (df_30days['datetime_utc'] >= t_start) &
        (df_30days['datetime_utc'] <= now_utc)
    ) if len(df_30days) > 0 else pd.Series([], dtype=bool)
    n_win = mask.sum() if len(df_30days) > 0 else 0

    _cm = _set_cmap_bad(CMAPS_TEMPORAL[i])
    hp.mollview(
        map_plot,
        cmap=_cm,
        title=f'{window_name}  (N={n_win:,})',
        unit='log10(1+N)',
        hold=True,
        sub=(2, 2, i + 1),
        margins=(0.01, 0.07, 0.01, 0.04),
        bgcolor=EMPTY_PIX,
        badcolor=EMPTY_PIX,
    )
    hp.graticule(dpar=30, dmer=60, alpha=0.70, color='#444444', lw=1.0, local=False)
    hp.graticule(dpar=10, dmer=30, alpha=0.22, color='#888888', lw=0.38, local=False)

    # Galactic plane
    idx = np.argsort(_gal_plane_ra)
    ra_s, dec_s = _gal_plane_ra[idx], _gal_plane_dec[idx]
    jumps = np.where(np.abs(np.diff(ra_s)) > 180)[0] + 1
    for seg_ra, seg_dec in zip(np.split(ra_s, jumps), np.split(dec_s, jumps)):
        hp.projplot(np.radians(90 - seg_dec), np.radians(seg_ra),
                    '-', lw=1.5, alpha=0.85, color='#0080ff', lonlat=False)

    # DDFs (markers only)
    for name, coord_obj in DDF.items():
        hp.projscatter(
            np.radians(90 - coord_obj.dec.deg), np.radians(coord_obj.ra.deg),
            marker='+', color='white', s=25,
            edgecolors='#ff7700', linewidths=0.5, zorder=6, lonlat=False,alpha=0.5
        )

plt.suptitle(
    f'Fink/LSST -- Alert temporal dynamics\n'
    f'(reference: {now_utc.strftime("%Y-%m-%d %H:%M UTC")})',
    fontsize=13, fontweight='bold', y=1.01
)
plt.savefig('fink_temporal_comparison.pdf', bbox_inches='tight', dpi=150)
plt.savefig('fink_temporal_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fink_temporal_comparison.pdf / .png')

## 8. Custom date range

In [ ]:
print(f'Custom range: {CUSTOM_START} -> {CUSTOM_STOP}')

t_custom_start = datetime.fromisoformat(CUSTOM_START).replace(tzinfo=timezone.utc)
t_custom_stop  = datetime.fromisoformat(CUSTOM_STOP).replace(tzinfo=timezone.utc)

t_available_start = now_utc - timedelta(days=30)

if t_custom_start >= t_available_start and t_custom_stop <= now_utc:
    print('Filtering from 30-day cache...')
    mask_custom = (
        (df_30days['datetime_utc'] >= t_custom_start) &
        (df_30days['datetime_utc'] <= t_custom_stop)
    ) if len(df_30days) > 0 else pd.Series([], dtype=bool)
    df_custom = df_30days[mask_custom] if len(df_30days) > 0 else df_30days
else:
    print('Range outside cache -> downloading directly from API...')
    df_custom = fetch_all_tags(
        startdate=CUSTOM_START,
        stopdate=CUSTOM_STOP,
        n=N_PER_TAG
    )

print(f'Alerts in custom range: {len(df_custom):,}')

sm_custom = plot_temporal_skymap(
    df_custom,
    title='Fink/LSST -- Custom date range',
    cmap='viridis',
    log_scale=(len(df_custom) > 10),
    start_str=CUSTOM_START,
    stop_str=CUSTOM_STOP,
)

## 9. Differential skymaps by alert age

In [ ]:
t_24h  = now_utc - timedelta(days=1)
t_3d   = now_utc - timedelta(days=3)
t_7d   = now_utc - timedelta(days=7)
t_30d  = now_utc - timedelta(days=30)

def time_mask(df, t_start, t_stop):
    if len(df) == 0:
        return df
    return df[(df['datetime_utc'] >= t_start) & (df['datetime_utc'] < t_stop)]

layers = {
    'Last 24h'         : time_mask(df_30days, t_24h, now_utc),
    '24h - 3 days'     : time_mask(df_30days, t_3d,  t_24h),
    '3 days - 1 week'  : time_mask(df_30days, t_7d,  t_3d),
    '1 week - 1 month' : time_mask(df_30days, t_30d, t_7d),
}

fig, _ = plt.subplots(2, 2, figsize=(22, 12))
plt.subplots_adjust(hspace=0.01, wspace=0.0)
cmaps_diff = ['Reds', 'Oranges', 'Blues', 'Greens']

for i, (lname, df_l) in enumerate(layers.items()):
    plt.subplot(2, 2, i + 1)
    sm_l = make_skymap_from_df(df_l)
    map_plot = np.log10(1 + sm_l)
    map_plot[map_plot == 0] = hp.UNSEEN

    _cm = _set_cmap_bad(cmaps_diff[i])
    hp.mollview(
        map_plot,
        cmap=_cm,
        title=f'{lname}  (N={len(df_l):,})',
        unit='log10(1+N)',
        hold=True,
        sub=(2, 2, i + 1),
        margins=(0.01, 0.07, 0.01, 0.04),
        bgcolor=EMPTY_PIX,
        badcolor=EMPTY_PIX,
    )
    hp.graticule(dpar=30, dmer=60, alpha=0.70, color='#444444', lw=1.0, local=False)
    hp.graticule(dpar=10, dmer=30, alpha=0.22, color='#888888', lw=0.38, local=False)

    # Galactic plane
    idx = np.argsort(_gal_plane_ra)
    ra_s, dec_s = _gal_plane_ra[idx], _gal_plane_dec[idx]
    jumps = np.where(np.abs(np.diff(ra_s)) > 180)[0] + 1
    for seg_ra, seg_dec in zip(np.split(ra_s, jumps), np.split(dec_s, jumps)):
        hp.projplot(np.radians(90 - seg_dec), np.radians(seg_ra),
                    '-', lw=1.5, alpha=0.85, color='#0080ff', lonlat=False)

    # DDFs (markers only)
    for name, coord_obj in DDF.items():
        hp.projscatter(
            np.radians(90 - coord_obj.dec.deg), np.radians(coord_obj.ra.deg),
            marker='+', color='white', s=25,
            edgecolors='#ff7700', linewidths=0.5, zorder=6, lonlat=False,alpha=0.5
        )

plt.suptitle(
    'Fink/LSST -- Alert distribution by age bracket\n'
    f'(as of {now_utc.strftime("%Y-%m-%d %H:%M UTC")})',
    fontsize=13, fontweight='bold', y=1.01
)
plt.savefig('fink_temporal_layers.pdf', bbox_inches='tight', dpi=150)
plt.savefig('fink_temporal_layers.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fink_temporal_layers.pdf / .png')

## 10. Animation: 3-day sliding window over 30 days

In [ ]:
from matplotlib import animation
from IPython.display import HTML

WINDOW_DAYS = 3    # sliding window width
STEP_DAYS   = 1    # step size

frames_data = []
t_cursor = t_30d
while t_cursor + timedelta(days=WINDOW_DAYS) <= now_utc:
    t_end = t_cursor + timedelta(days=WINDOW_DAYS)
    df_f  = time_mask(df_30days, t_cursor, t_end)
    sm_f  = make_skymap_from_df(df_f)
    frames_data.append({
        'skymap':    sm_f,
        'n_alerts':  len(df_f),
        't_start':   t_cursor.strftime('%Y-%m-%d'),
        't_end':     t_end.strftime('%Y-%m-%d'),
    })
    t_cursor += timedelta(days=STEP_DAYS)

print(f'{len(frames_data)} frames generated for animation.')

if len(frames_data) == 0:
    print('Not enough data for animation.')
else:
    global_max = max(
        np.log10(1 + f['skymap']).max() for f in frames_data
    )

    fig = plt.figure(figsize=(14, 7))

    def animate(i):
        plt.clf()
        fd = frames_data[i]
        map_plot = np.log10(1 + fd['skymap'])
        map_plot[map_plot == 0] = hp.UNSEEN

        _cm = _set_cmap_bad('hot')
        hp.mollview(
            map_plot,
            cmap=_cm,
            title=f'Window {fd["t_start"]} -> {fd["t_end"]}  '
                  f'(N={fd["n_alerts"]:,})',
            unit='log10(1+N)',
            min=0, max=global_max,
            hold=False,
            margins=(0.01, 0.06, 0.01, 0.04),
            bgcolor=EMPTY_PIX,
            badcolor=EMPTY_PIX,
        )
        hp.graticule(dpar=30, dmer=60, alpha=0.70, color='#444444', lw=1.0, local=False)
        hp.graticule(dpar=10, dmer=30, alpha=0.22, color='#888888', lw=0.38, local=False)

        # Galactic plane
        idx = np.argsort(_gal_plane_ra)
        ra_s, dec_s = _gal_plane_ra[idx], _gal_plane_dec[idx]
        jumps = np.where(np.abs(np.diff(ra_s)) > 180)[0] + 1
        for seg_ra, seg_dec in zip(np.split(ra_s, jumps), np.split(dec_s, jumps)):
            hp.projplot(np.radians(90 - seg_dec), np.radians(seg_ra),
                        '-', lw=1.0, alpha=0.7, color='cyan', lonlat=False)

        # DDFs
        for name, coord_obj in DDF.items():
            hp.projscatter(
                np.radians(90 - coord_obj.dec.deg), np.radians(coord_obj.ra.deg),
                marker='+', color='white', s=30,
                edgecolors='orange', linewidths=0.5, zorder=6, lonlat=False,alpha=0.5
            )

        plt.figtext(0.12, 0.02,
                    f'Sliding window {WINDOW_DAYS}d  |  NSIDE={NSIDE}',
                    fontsize=8, color='gray')

    anim = animation.FuncAnimation(
        fig, animate, frames=len(frames_data),
        interval=400, repeat=True
    )

    writer = animation.PillowWriter(fps=3)
    anim.save('fink_temporal_animation.gif', writer=writer, dpi=100)
    print('Animation saved: fink_temporal_animation.gif')

    plt.close()
    HTML(anim.to_jshtml())

## 11. Temporal evolution of alert rate

In [ ]:
if len(df_30days) > 0:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    df_sorted = df_30days.sort_values('datetime_utc')
    df_sorted['hour_bin'] = df_sorted['datetime_utc'].dt.floor('1h')
    hourly = df_sorted.groupby('hour_bin').size().reset_index(name='count')

    axes[0].bar(hourly['hour_bin'], hourly['count'],
                width=0.04, color='steelblue', alpha=0.8, label='All alerts')
    rolling = hourly.set_index('hour_bin')['count'].rolling('24h').mean()
    axes[0].plot(rolling.index, rolling.values, 'r-', lw=1.5, label='24h rolling mean')
    axes[0].set_ylabel('Alerts / hour')
    axes[0].set_title('Fink hourly alert rate (last 30 days)')
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

    for tag in TAGS:
        df_t = df_sorted[df_sorted['tag'] == tag].copy()
        if len(df_t) == 0:
            continue
        df_t['day_bin'] = df_t['datetime_utc'].dt.floor('1D')
        daily_t = df_t.groupby('day_bin').size().reset_index(name='count')
        short_tag = tag.replace('_candidate', '').replace('_', ' ')
        axes[1].plot(daily_t['day_bin'], daily_t['count'],
                     marker='o', ms=4, lw=1.5, label=short_tag)

    axes[1].set_xlabel('Date (UTC)')
    axes[1].set_ylabel('Alerts / day')
    axes[1].set_title('Daily rate per tag')
    axes[1].legend(fontsize=7, ncol=2)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('fink_alert_rate_30days.pdf', bbox_inches='tight', dpi=150)
    plt.show()
    print('Saved: fink_alert_rate_30days.pdf')
else:
    print('No data available for temporal plot.')